# SARBLOH wheels: offline wheelhouse builder

The RTX Pro 6000 submission runs with **internet off**, so every package beyond the Kaggle image has to arrive as
wheels in an attached input. This notebook runs **with internet on**. It downloads the pinned serving stack, writes
`requirements.lock`, proves the wheelhouse installs offline and imports, then leaves it in
`/kaggle/working/wheelhouse`.

**How to use**
1. Settings: Internet **on**. Accelerator: the same GPU family as the submission if quota allows, otherwise
   a T4 is fine for building; the GPU check is informational. Save Version, then "Save & Run All".
2. Attach this notebook's **output** to `main.ipynb` (Add Input, then Notebook Output; it mounts at
   `/kaggle/input/notebooks/<user>/<slug>/wheelhouse`), or make a dataset from the output.
3. In `main.ipynb`, set `CONFIG["wheelhouse_dataset"]` to it and set `CONFIG["wheelhouse_stamp"]` to the
   `STAMP` printed at the end.

The layout matches `driessmit1/arc3-vllm-h100-wheelhouse-v3` (wheels plus `requirements.lock`), so `main.ipynb`
installs either one with the same command. `arc-agi` is **not** built here: it comes from the competition mount
(`/kaggle/input/competitions/arc-prize-2026-arc-agi-3/arc_agi_3_wheels`).

## CONFIG

In [ ]:
CONFIG = {
    # Pinned to the stack the Duck proved on the RTX Pro 6000. Bump deliberately and record the bump (new STAMP).
    "packages": ["vllm==0.19.0", "torch==2.10.0", "flashinfer-python==0.6.6", "transformers==4.57.6"],  # = driessmit1 v3
    # Tried one by one; a missing optional package is reported, not fatal.
    "optional_packages": ["flashinfer-cubin==0.6.6"],   # prebuilt kernels: avoids flashinfer JIT at start-up
    "extra_index_urls": [],                             # e.g. a torch CUDA-specific index if a pin requires it
    "out_dir": "/kaggle/working/wheelhouse",
}

## 1. Build environment (the wheels must match the submission image: same Python, same glibc)

In [ ]:
import json, os, platform, shutil, subprocess, sys, time
from pathlib import Path

OUT = Path(CONFIG["out_dir"])
print("python", sys.version.split()[0], "|", platform.platform(), "| glibc", platform.libc_ver())
print(shutil.which("nvidia-smi") and subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total,driver_version",
                     "--format=csv"], capture_output=True, text=True).stdout or "no GPU (fine for building)")


def pip(*args, check=True):
    cmd = [sys.executable, "-m", "pip", *args, "--disable-pip-version-check"]
    for url in CONFIG["extra_index_urls"]:
        cmd += ["--extra-index-url", url]
    print("$ pip", " ".join(args), flush=True)
    return subprocess.run(cmd, check=check)

## 2. Download every wheel the pinned packages resolve to

In [ ]:
shutil.rmtree(OUT, ignore_errors=True)
OUT.mkdir(parents=True)
t = time.time()
pip("download", "--dest", str(OUT), "--only-binary", ":all:", *CONFIG["packages"])
optional_ok = []
for pkg in CONFIG["optional_packages"]:
    if pip("download", "--dest", str(OUT), "--only-binary", ":all:", pkg, check=False).returncode == 0:
        optional_ok.append(pkg)
    else:
        print(f"optional package unavailable, skipped: {pkg}")
wheels = sorted(OUT.glob("*.whl"))
size_gb = sum(w.stat().st_size for w in wheels) / 1e9
print(f"{len(wheels)} wheels, {size_gb:.1f} GB, {time.time() - t:.0f}s")

## 3. Write `requirements.lock`: the exact resolved set, resolved **offline** from the wheelhouse only

In [ ]:
report = Path("/tmp/resolve-report.json")
pip("install", "--dry-run", "--ignore-installed", "--no-index", "--find-links", str(OUT),
    "--report", str(report), "--quiet", *CONFIG["packages"], *optional_ok)
items = json.loads(report.read_text())["install"]
lock = sorted(f"{i['metadata']['name']}=={i['metadata']['version']}" for i in items)
(OUT / "requirements.lock").write_text("\n".join(lock) + "\n")
print(f"requirements.lock: {len(lock)} pins")
print("\n".join(l for l in lock if l.split("==")[0].lower() in {"vllm", "torch", "flashinfer-python", "transformers", "triton"}))

## 4. Prove it: install offline exactly as `main.ipynb` does, then import and check CUDA

In [ ]:
target = Path("/tmp/wheelhouse-verify")
shutil.rmtree(target, ignore_errors=True)
t = time.time()
subprocess.run([sys.executable, "-m", "pip", "install", "--no-index", "--find-links", str(OUT),
                "--requirement", str(OUT / "requirements.lock"), "--target", str(target), "--upgrade",
                "--ignore-installed", "--only-binary", ":all:", "--no-compile", "--disable-pip-version-check",
                "--no-warn-conflicts", "--quiet"], check=True)
print(f"offline install ok in {time.time() - t:.0f}s")
env = dict(os.environ, PYTHONPATH=str(target), VLLM_NO_USAGE_STATS="1")
probe = ("import vllm, torch; print('vllm', vllm.__version__, '| torch', torch.__version__, '| cuda', torch.version.cuda, "
         "'| gpu', torch.cuda.is_available() and torch.cuda.get_device_name(0))")
res = subprocess.run([sys.executable, "-c", probe], env=env, capture_output=True, text=True)
print(res.stdout, res.stderr[-3000:])
assert res.returncode == 0, "import failed: the wheelhouse is not usable"
shutil.rmtree(target, ignore_errors=True)   # keep the notebook output to the wheelhouse only

## 5. Stamp: paste into `main.ipynb` `CONFIG["wheelhouse_stamp"]`

In [ ]:
pins = {l.split("==")[0].lower(): l for l in (OUT / "requirements.lock").read_text().split()}
STAMP = " ".join(pins[p] for p in ("vllm", "torch", "flashinfer-python") if p in pins) + "\n"
(OUT / "STAMP").write_text(STAMP)
manifest = {"built_at": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()), "python": sys.version.split()[0],
            "packages": CONFIG["packages"], "optional": optional_ok, "wheels": len(wheels), "size_gb": round(size_gb, 2)}
(OUT / "manifest.json").write_text(json.dumps(manifest, indent=1))
print("STAMP =", repr(STAMP))
print(json.dumps(manifest, indent=1))